**Key Deliverables**
- TextCleaner class with 6+ normalization methods 
- Abbreviation dictionary with 30+ mappings 
- Test suite with 40+ test cases covering edge cases 
- Cleaned dataset generated from Week 1 sample 
- Before/after examples showing improvements 
- NEW: Data profiling report showing what needs cleaning (null rates, HTML 
presence, common abbreviations)

In [1]:
import re
import pandas as pd
from collections import Counter

In [2]:
df = pd.read_csv("../data/processed/listing_sample.csv")

In [3]:
df.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks
0,1169503734,133 Crystal Street,Taft,3.0,2.0,235000,This unique property offers two homes on one l...
1,1154220129,15664 Kadota,Victorville,4.0,4.0,459000,Beautiful Two-Story Home in Victorville – Spac...
2,1159921951,949 S Manhattan Place 203,Los Angeles,3.0,2.0,689000,Presenting this exquisite second-floor corner ...
3,1170333192,1872 Seascape Boulevard,Aptos,3.0,3.0,1195000,Thoughtfully renovated coastal retreat tucked ...
4,1152463169,2085 Westhampton Drive,Arroyo Grande,3.0,2.0,1050000,Welcome to 2085 Westhampton Drive in Arroyo Gr...


In [11]:
df.dtypes

L_ListingID      int64
L_Address          str
L_City             str
beds           float64
baths          float64
price            int64
remarks            str
dtype: object

In [4]:
df['L_City'].value_counts()

L_City
Los Angeles     63
San Diego       31
San Jose        27
Irvine          17
Palm Desert     14
                ..
Long Barn        1
Oxnard           1
Fresno           1
North Tustin     1
Encino           1
Name: count, Length: 341, dtype: int64

In [5]:
class TextCleaner: 
    def __init__(self): 
        self.abbrev_map = { 
        'br': 'bedroom', 'ba': 'bathroom', 'sqft': 'square feet', 
        'w/': 'with', 'w/o': 'without', 'mbr': 'master bedroom' 
    } 
    def clean_text(self, text): 
        text = self.normalize_unicode(text) 
        text = self.normalize_prices(text) 
        text = self.normalize_measurements(text) 
        text = self.expand_abbreviations(text) 
        return text.strip() 
    def normalize_prices(self, text): 
        # 450k → 450000 
        text = re.sub(r'(\d+)k', lambda m: str(int(m.group(1))*1000), text, 
        flags=re.I) 
        # 1.2m → 1200000 
        text = re.sub(r'(\d+\.?\d*)m', lambda m: 
        str(int(float(m.group(1))*1000000)), text, flags=re.I) 
        return text 
    def _extract_top_ngrams(self, series, n=20):
        words = []
        for text in series.dropna(): 
            words.extend(re.findall(r'\b\w+\b', text.lower()))
        return Counter(words).most_common(n)
    def _detect_abbreviations(self, series):
        counts = Counter()
        for text in series.dropna():
            words = re.findall(r'\b\w+\b', text.lower())
            for word in words:
                if word in self.abbrev_map:
                    counts[word] += 1
        return counts.most_common(10)
    
    # Before cleaning, understand what needs cleaning
    def profile_column(self, df, column_name): 
        """Analyze what's actually in L_Remarks""" 
        return { 
            'null_rate': df[column_name].isnull().mean(), 
            'avg_length': df[column_name].str.len().mean(), 
            'common_terms': self._extract_top_ngrams(df[column_name]), 
            'price_mentions': df[column_name].str.contains(r'\$\d').sum(), 
            'has_html': df[column_name].str.contains('<').sum(), 
            'common_abbreviations': self._detect_abbreviations(df[column_name]) 
        } 

#### 1. Data Profiling (Learn the data, what is in there, what kind of noises)

Inside the profile_column():

A. Determine the % null/empty strings, % extremely short vs. long remarks, duplicate remarks frequency

B. Measure the HTML presence, any unicode noises like smart quotes, weird dashes, emojis, etc., the different price formats (e.g. $450,000; 450k, 1.2m), measurement formats (e.g. 2,000 sqft, 2000 SF, 2k sq ft), and abbreviations frequency (e.g. br, ba, mbr, w/, etc.). 

In [10]:
cleaner = TextCleaner()

In [9]:
profile_remarks = cleaner.profile_column(df, 'remarks') 
print(f"Null rate: {profile_remarks['null_rate']:.2%}")
print(f"Average length: {profile_remarks['avg_length']:.2f} characters")
print(f"Price mentions: {profile_remarks['price_mentions']} listings")
print(f"HTML tags found in {profile_remarks['has_html']} listings") 
print(f"Common abbreviations: {profile_remarks['common_abbreviations']}")

Null rate: 0.00%
Average length: 1290.09 characters
Price mentions: 57 listings
HTML tags found in 1 listings
Common abbreviations: [('sqft', 32), ('ba', 4), ('br', 3)]


In the remarks, we see that there are 57 out of the 1000 entries that mentioned about price. Let's look at those specified entries to determine the different price formats

In [12]:
profile_remarks['price_mentions']

np.int64(57)